[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/coding_eval_braintrust/coding_eval.ipynb)

# Coding eval: Braintrust + Fireworks

Measure **vanilla** (base) model quality on Python function completion — before fine-tuning or agent work.

We use two tiers:

| Tier | Dataset | Rows | When |
| --- | --- | ---: | --- |
| **1 — Smoke** | Bundled `smoke_data.jsonl` | 20 | First run: confirm keys, model id, scoring |
| **2 — Standard** | HuggingFace [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval) | 164 | Baseline pass@1 |

**Stack**

- **Inference:** Fireworks serverless models via the [Braintrust gateway](https://www.braintrust.dev/docs/integrations/ai-providers/fireworks) (auto-tracing) or direct Fireworks API
- **Eval loop:** Braintrust `Eval` — dataset + task + scorers
- **Scoring:** deterministic test execution (HumanEval-style), not an LLM judge

## Prerequisites

```bash
pip install braintrust openai datasets python-dotenv
```

`.env` at repo root (or export manually):

```bash
FIREWORKS_API_KEY=...
BRAINTRUST_API_KEY=...
```

When using the Braintrust gateway, also add your Fireworks key as an [org/project AI provider](https://www.braintrust.dev/docs/admin/ai-providers) in Braintrust.

> **Cost:** tier-1 is ~20 completions; tier-2 is 164. Keep `HUMANEVAL_LIMIT` small on first pass.

## 1. Setup

In [ ]:
# %pip install -q braintrust openai datasets python-dotenv

import json
import os
import sys
from pathlib import Path

import dotenv
from openai import OpenAI

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)

CASE_DIR = Path.cwd()
if not (CASE_DIR / "coding_sandbox.py").exists():
    # Walk up from notebook cwd until we find this case-study folder.
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "coding_eval_braintrust" / "coding_sandbox.py").exists():
            CASE_DIR = parent / "coding_eval_braintrust"
            break
assert (CASE_DIR / "coding_sandbox.py").exists(), "could not locate coding_eval_braintrust/"
sys.path.insert(0, str(CASE_DIR))

from coding_sandbox import build_program, passes_tests, run_python_tests

assert os.getenv("FIREWORKS_API_KEY"), "set FIREWORKS_API_KEY"
assert os.getenv("BRAINTRUST_API_KEY"), "set BRAINTRUST_API_KEY"

print("case dir:", CASE_DIR)

## 2. Configuration

Pick one model and how much of HumanEval to run. Start with tier-1 only (`RUN_TIER_1=True`, `RUN_TIER_2=False`).

In [ ]:
# CONFIG — edit these
USE_BRAINTRUST_GATEWAY = True  # False -> call Fireworks API directly

# Gateway ids: fireworks/<short-name> (see Braintrust Fireworks docs)
# Direct API:  accounts/fireworks/models/<model-id>
MODEL = "fireworks/qwen3-8b" if USE_BRAINTRUST_GATEWAY else "accounts/fireworks/models/qwen3-8b"

BRAINTRUST_PROJECT = "fireworks-coding-eval"
EXPERIMENT_PREFIX = MODEL.rsplit("/", 1)[-1]

TEMPERATURE = 0.0
MAX_TOKENS = 512
TEST_TIMEOUT_S = 5.0

RUN_TIER_1 = True   # 20-row smoke set (bundled)
RUN_TIER_2 = False  # HumanEval — flip on after smoke passes
HUMANEVAL_LIMIT = 30  # set to 164 for full split; keep small on first pass

SYSTEM_PROMPT = (
    "You are a Python expert. Complete the function body for the given signature. "
    "Return only valid Python code — no markdown fences, no explanation."
)

if USE_BRAINTRUST_GATEWAY:
    client = OpenAI(
        base_url="https://gateway.braintrust.dev/v1",
        api_key=os.environ["BRAINTRUST_API_KEY"],
    )
else:
    client = OpenAI(
        base_url="https://api.fireworks.ai/inference/v1",
        api_key=os.environ["FIREWORKS_API_KEY"],
    )

print("model:", MODEL)
print("gateway:", USE_BRAINTRUST_GATEWAY)

## 3. Load datasets

Each row is HumanEval-shaped: `prompt` (function stub), `test` (asserts), `entry_point`.

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def to_eval_rows(raw_rows: list[dict]) -> list[dict]:
    """Braintrust rows: input + expected."""
    return [
        {
            "input": {
                "task_id": r["task_id"],
                "prompt": r["prompt"],
                "entry_point": r["entry_point"],
            },
            "expected": {"test": r["test"]},
        }
        for r in raw_rows
    ]


smoke_rows = to_eval_rows(load_jsonl(CASE_DIR / "smoke_data.jsonl"))
print(f"tier-1 smoke: {len(smoke_rows)} rows")

humaneval_rows: list[dict] = []
if RUN_TIER_2:
    from datasets import load_dataset

    ds = load_dataset("openai/openai_humaneval", split="test")
    if HUMANEVAL_LIMIT:
        ds = ds.select(range(min(HUMANEVAL_LIMIT, len(ds))))
    humaneval_rows = to_eval_rows([dict(r) for r in ds])
    print(f"tier-2 humaneval: {len(humaneval_rows)} rows")

## 4. Task + scorer

The **task** calls Fireworks and returns raw completion text. The **scorer** executes bundled unit tests in a subprocess (`coding_sandbox.py`).

In [ ]:
def complete_function(prompt: str, *, model: str | None = None) -> str:
    model = model or MODEL
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


def coding_task(row: dict) -> str:
    return complete_function(row["prompt"])


def test_pass_scorer(output, expected, input, **kwargs):
    return passes_tests(output, expected, input=input)

### Quick sanity check (no Braintrust)

Run one smoke row locally before launching a full experiment.

In [ ]:
_demo = smoke_rows[0]
_demo_out = coding_task(_demo["input"])
_demo_prog = build_program(_demo["input"]["prompt"], _demo_out)
_demo_ok, _demo_err = run_python_tests(_demo_prog, _demo["expected"]["test"], timeout=TEST_TIMEOUT_S)
print("task:", _demo["input"]["task_id"])
print("passed:", _demo_ok)
if _demo_err:
    print("error:", _demo_err[:500])
print("--- model output (first 400 chars) ---")
print(_demo_out[:400])

## 5. Tier 1 — smoke eval

Runs all 20 bundled rows through Braintrust. Open the printed experiment URL to inspect failures row-by-row.

In [ ]:
from braintrust import Eval, init_logger

init_logger(project=BRAINTRUST_PROJECT)

tier1_result = None
if RUN_TIER_1:
    tier1_result = Eval(
        "fireworks-coding-smoke",
        data=smoke_rows,
        task=coding_task,
        scores=[test_pass_scorer],
        metadata={"model": MODEL, "tier": 1},
        experiment_name=f"{EXPERIMENT_PREFIX}-smoke",
        max_concurrency=4,
    )
    print("tier-1 summary:", tier1_result.summary if tier1_result else None)
else:
    print("skipped tier-1")

## 6. Tier 2 — HumanEval

Flip `RUN_TIER_2=True` in the config cell. Increase `HUMANEVAL_LIMIT` toward 164 for a full baseline.

In [ ]:
tier2_result = None
if RUN_TIER_2:
    tier2_result = Eval(
        "fireworks-coding-humaneval",
        data=humaneval_rows,
        task=coding_task,
        scores=[test_pass_scorer],
        metadata={"model": MODEL, "tier": 2, "limit": HUMANEVAL_LIMIT},
        experiment_name=f"{EXPERIMENT_PREFIX}-humaneval-{HUMANEVAL_LIMIT}",
        max_concurrency=4,
    )
    print("tier-2 summary:", tier2_result.summary if tier2_result else None)
else:
    print("skipped tier-2 (set RUN_TIER_2=True to enable)")

## 7. (Optional) Compare multiple models

Run the same tier-1 smoke set across a short model list. Each model gets its own Braintrust experiment — compare them side-by-side in the UI.

In [ ]:
COMPARE_MODELS = False  # flip to True after smoke works

MODELS_TO_COMPARE = [
    "fireworks/qwen3-8b",
    "fireworks/deepseek-v3p2",
]

if COMPARE_MODELS:
    summaries = {}
    for m in MODELS_TO_COMPARE:
        result = Eval(
            "fireworks-coding-smoke-compare",
            data=smoke_rows,
            task=lambda row, model=m: complete_function(row["prompt"], model=model),
            scores=[test_pass_scorer],
            metadata={"model": m, "tier": 1},
            experiment_name=f"{m.rsplit('/', 1)[-1]}-smoke",
            max_concurrency=4,
        )
        summaries[m] = result.summary
    print(summaries)
else:
    print("set COMPARE_MODELS=True to run model comparison")

## Next steps

- **Drill into failures** in the Braintrust UI — filter rows where `passes_tests` = 0.
- **Lock inference settings** (`temperature`, prompt, `max_tokens`) before comparing models.
- **Add your own rows** to `smoke_data.jsonl` for product-specific snippets before scaling to HumanEval.
- **MBPP / LiveCodeBench** follow the same pattern: load from HuggingFace, map to `{prompt, test}`, reuse `test_pass_scorer`.
- **After baseline:** use Fireworks SFT/RFT cookbooks to improve the model, then re-run this notebook on the tuned checkpoint.